In [4]:
import os
import sys

import pandas as pd
import numpy as np

work_dir = "/home/handb/GeoSTHN"

os.chdir(work_dir)
sys.path.append(work_dir)

In [ ]:
import pandas as pd
import numpy as np

dataset = "thgl-github-subset"

# ================= 配置区 =================
EDGE_FILE = f"tgb/DATA/{dataset.replace('-', '_')}/{dataset}_edgelist.csv"
NODE_FILE = f"tgb/DATA/{dataset.replace('-', '_')}/{dataset}_nodetype.csv"
# =========================================
t_col = "timestamp"
src_col = "head"
dst_col = "tail"
rel_col = "relation_type"

print("🔄 正在加载数据...")

# 1. 加载边列表
# 假设CSV没有表头，通常TGB格式为: [timestamp, source_id, destination_id, edge_type/weight]
# 如果你的文件有表头，请将 header=None 改为 header=0
df_edges = pd.read_csv(EDGE_FILE)
df_edges = df_edges.rename(
    columns={
        t_col: "ts",
        src_col: "src",
        dst_col: "dst",
        rel_col: "rel",
    }
)

# 2. 加载节点类型映射
df_nodes = pd.read_csv(NODE_FILE)
node_type_map = dict(zip(df_nodes["node_id"], df_nodes["type"]))

print(f"✅ 数据加载完毕: {len(df_edges)} 条边, {len(df_nodes)} 个节点")

# ================= 核心逻辑：寻找“有故事”的节点 =================

# 1. 划分时间段 (前20% vs 后20%)
t_min, t_max = df_edges["ts"].min(), df_edges["ts"].max()
duration = t_max - t_min
t_split_early = t_min + duration * 0.2
t_split_late = t_max - duration * 0.2

print(f"🕒 时间跨度: {t_min:.1f} -> {t_max:.1f}")

# 2. 统计每个节点在不同阶段的度 (交互次数)
# 这里我们将 src 和 dst 都视为交互参与者
all_nodes = pd.concat([df_edges["src"], df_edges["dst"]])
total_degree = all_nodes.value_counts()

# 早期阶段 (Early Stage)
early_edges = df_edges[df_edges["ts"] <= t_split_early]
early_nodes = pd.concat([early_edges["src"], early_edges["dst"]])
early_degree = early_nodes.value_counts()

# 晚期阶段 (Late Stage)
late_edges = df_edges[df_edges["ts"] >= t_split_late]
late_nodes = pd.concat([late_edges["src"], late_edges["dst"]])
late_degree = late_nodes.value_counts()

# ================= 筛选策略 =================

candidates = []

# 策略 A: "超级中枢" (Global Hubs) - 总交互量最高
print("\n🏆 --- 候选人榜单 A: 超级中枢 (总活跃度最高) ---")
top_hubs = total_degree.head(5).index.tolist()
for nid in top_hubs:
    ntype = node_type_map.get(nid, "Unknown")
    count = total_degree[nid]
    print(f"  👉 ID: {nid} | 类型: {ntype} | 总交互: {count}")
    candidates.append(nid)

# 策略 B: "后期之秀" (Rising Stars) - 后期活跃度远大于早期
print("\n🚀 --- 候选人榜单 B: 后期之秀 (爆发式增长) ---")
# 计算增长量：晚期度 - 早期度 (填充0以防报错)
growth_score = late_degree.subtract(early_degree, fill_value=0)
top_rising = growth_score.sort_values(ascending=False).head(5).index.tolist()

for nid in top_rising:
    ntype = node_type_map.get(nid, "Unknown")
    late_c = late_degree.get(nid, 0)
    early_c = early_degree.get(nid, 0)
    # 过滤掉那些本来就很强只是变得更强的，我们想要“从无到有”的
    if early_c < 50:
        print(
            f"  👉 ID: {nid} | 类型: {ntype} | 早期: {early_c} -> 晚期: {late_c} (增长: +{int(late_c - early_c)})"
        )
        candidates.append(nid)

print("\n💡 建议：")
print("1. 如果你想展示模型捕捉'层次结构'的能力，请从榜单 A 中选一个 Repo 节点。")
print("2. 如果你想展示模型捕捉'动态演化'的能力，请从榜单 B 中选一个增长最显著的节点。")

🔄 正在加载数据...
✅ 数据加载完毕: 50000 条边, 36818 个节点
🕒 时间跨度: 1709269200.0 -> 1709278348.0

🏆 --- 候选人榜单 A: 超级中枢 (总活跃度最高) ---
  👉 ID: 38 | 类型: 2 | 总交互: 3573
  👉 ID: 142 | 类型: 2 | 总交互: 2121
  👉 ID: 8 | 类型: 2 | 总交互: 1646
  👉 ID: 58 | 类型: 2 | 总交互: 1355
  👉 ID: 976 | 类型: 2 | 总交互: 307

🚀 --- 候选人榜单 B: 后期之秀 (爆发式增长) ---
  👉 ID: 31855 | 类型: 2 | 早期: 0 -> 晚期: 54 (增长: +54)
  👉 ID: 515 | 类型: 2 | 早期: 8 -> 晚期: 50 (增长: +42)
  👉 ID: 31678 | 类型: 1 | 早期: 0 -> 晚期: 32 (增长: +32)
  👉 ID: 24870 | 类型: 1 | 早期: 0 -> 晚期: 27 (增长: +27)

💡 建议：
1. 如果你想展示模型捕捉'层次结构'的能力，请从榜单 A 中选一个 Repo 节点。
2. 如果你想展示模型捕捉'动态演化'的能力，请从榜单 B 中选一个增长最显著的节点。
